In [1]:
%load_ext autoreload
%autoreload 2

In [6]:

from bokeh.layouts import row, column
from bokeh.models import Button, WMTSTileSource, ColumnDataSource, Slider, CustomJS, HoverTool, CheckboxGroup, ColorPicker, TextInput, Div, AutocompleteInput
from bokeh.plotting import figure, show

import os


from manager import SourceManager, DynamicSourceManager

import sys
sys.path.insert(0, '..')
from src.load_data import DataSet
from src.tiles import ImageTiles
from src.utils.bins2spots import Bins2Spots
from src.sample import Bins, Spots

In [3]:
from bokeh.io import output_notebook
output_notebook()

Loading BokehJS ...

In [105]:
dir_base = '/home/lalonso/PhD/VisHD/data/'
dataset = DataSet(dir_base=dir_base, img_file="Visium_HD_Mouse_Small_Intestine_tissue_image.btf", bins_size=16)
img, adata, df_tissue_positions = dataset.load()
print("Data loaded succesfully")

/home/lalonso/PhD/VisHD/.venv/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/lalonso/PhD/VisHD/.venv/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Data loaded succesfully


In [34]:
image_tiles = ImageTiles(img, os.path.join(dir_base, "./tiles/"), os.path.join(dir_base, "./StarDist/"))
port = 4000
image_tiles.deploy_server(port=port)
print(f"Tiles server deployed on port {port}")

In [122]:
b2s = Bins2Spots(adata, df_tissue_positions, bins_size=dataset.bins_size)

In [107]:
polys = image_tiles.segment_image()[1]
cells_gdf = image_tiles.get_cells_gdf(polys)

Loaded labels and polys from disk.


100%|██████████| 369005/369005 [00:27<00:00, 13457.97it/s]


In [29]:
from src.utils.operations import Pooler
pooler = Pooler(
    b2s.bin_squares, 
    b2s.adata_hd, 
    b2s.spot_polygons, 
    keep_cols=['array_row', 'array_col', 'pxl_row_in_fullres', 'pxl_col_in_fullres']
)

In [30]:
pooler.join()

,index_left,in_tissue,bins_index,gdf_index,geometry,array_row,array_col,pxl_row_in_fullres,pxl_col_in_fullres
s_016um_00418_00017-1,s_016um_00418_00017-1,0.0,s_016um_00418_00017-1,0,"POLYGON ((200.846 100.423, 200.363 90.580, 198...",0,0,24250.902665,100.423175
s_016um_00418_00016-1,s_016um_00418_00016-1,0.0,s_016um_00418_00016-1,0,"POLYGON ((200.846 100.423, 200.363 90.580, 198...",0,0,24250.902665,100.423175
s_016um_00418_00015-1,s_016um_00418_00015-1,0.0,s_016um_00418_00015-1,0,"POLYGON ((200.846 100.423, 200.363 90.580, 198...",0,0,24250.902665,100.423175
s_016um_00417_00016-1,s_016um_00417_00016-1,0.0,s_016um_00417_00016-1,0,"POLYGON ((200.846 100.423, 200.363 90.580, 198...",0,0,24250.902665,100.423175
s_016um_00418_00022-1,s_016um_00418_00022-1,0.0,s_016um_00418_00022-1,1,"POLYGON ((566.022 100.423, 565.538 90.580, 564...",0,2,24250.902665,465.598358
...,...,...,...,...,...,...,...,...,...
s_016um_00004_00414-1,s_016um_00004_00414-1,0.0,s_016um_00004_00414-1,5156,"POLYGON ((23572.058 24135.498, 23571.574 24125...",76,128,215.827810,23471.634859
s_016um_00004_00413-1,s_016um_00004_00413-1,0.0,s_016um_00004_00413-1,5156,"POLYGON ((23572.058 24135.498, 23571.574 24125...",76,128,215.827810,23471.634859
s_016um_00003_00413-1,s_016um_00003_00413-1,0.0,s_016um_00003_00413-1,5156,"POLYGON ((23572.058 24135.498, 23571.574 24125...",76,128,215.827810,23471.634859
NaN,NaN,NaN,NaN,5157,"POLYGON ((23937.233 24135.498, 23936.750 24125...",76,130,215.827810,23836.810042


In [31]:
pooler.filter().obs

,in_tissue,index,array_row,array_col,pxl_row_in_fullres,pxl_col_in_fullres,gdf_index
s_016um_00048_00157-1,1,s_016um_00048_00157-1,68,46,2745.835689,8499.452374,4579
s_016um_00048_00163-1,1,s_016um_00048_00163-1,68,48,2745.835689,8864.627557,4580
s_016um_00048_00164-1,1,s_016um_00048_00164-1,68,48,2745.835689,8864.627557,4580
s_016um_00048_00170-1,1,s_016um_00048_00170-1,68,50,2745.835689,9229.802739,4581
s_016um_00048_00176-1,1,s_016um_00048_00176-1,68,52,2745.835689,9594.977922,4582
...,...,...,...,...,...,...,...
s_016um_00411_00306-1,1,s_016um_00411_00306-1,1,93,23934.651680,17081.069164,113
s_016um_00411_00307-1,1,s_016um_00411_00307-1,1,93,23934.651680,17081.069164,113
s_016um_00411_00319-1,1,s_016um_00411_00319-1,1,97,23934.651680,17811.419530,115
s_016um_00416_00353-1,1,s_016um_00416_00353-1,0,108,24250.902665,19819.883034,54


In [32]:
pooler.pool()

/home/lalonso/PhD/VisHD/.venv/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 2837 × 19059
    obs: 'array_row', 'array_col', 'pxl_row_in_fullres', 'pxl_col_in_fullres'
    var: 'gene_ids', 'feature_types', 'genome'
    obsm: 'spatial'

In [123]:
from src.utils.operations import CellsPooler
pooler = CellsPooler(
    b2s.bin_squares, 
    b2s.adata_hd, 
    cells_gdf, 
    keep_cols=[]
)

175561it [00:06, 27931.98it/s]


In [124]:
pooler.join(predicate='intersects')

,index_left,in_tissue,array_row,array_col,pxl_row_in_fullres,pxl_col_in_fullres,bins_index,geometry,gdf_index
s_016um_00398_00052-1,s_016um_00398_00052-1,0,398,52,1243.065127,2208.312860,s_016um_00398_00052-1,"POLYGON ((2240.695 1216.258, 2240.044 1217.551...",ID_1
s_016um_00398_00053-1,s_016um_00398_00053-1,0,398,53,1242.676891,2266.745284,s_016um_00398_00053-1,"POLYGON ((2240.695 1216.258, 2240.044 1217.551...",ID_1
s_016um_00399_00052-1,s_016um_00399_00052-1,0,399,52,1184.634238,2207.923120,s_016um_00399_00052-1,"POLYGON ((2240.695 1216.258, 2240.044 1217.551...",ID_1
s_016um_00399_00053-1,s_016um_00399_00053-1,0,399,53,1184.246007,2266.355549,s_016um_00399_00053-1,"POLYGON ((2240.695 1216.258, 2240.044 1217.551...",ID_1
s_016um_00406_00081-1,s_016um_00406_00081-1,1,406,81,764.360044,3899.732429,s_016um_00406_00081-1,"POLYGON ((3917.366 764.197, 3917.390 764.923, ...",ID_2
...,...,...,...,...,...,...,...,...,...
s_016um_00081_00305-1,s_016um_00081_00305-1,0,81,305,19666.453865,17114.474184,s_016um_00081_00305-1,"POLYGON ((17114.328 19649.556, 17112.770 19652...",ID_369001
s_016um_00079_00303-1,s_016um_00079_00303-1,0,79,303,19784.085463,16998.394648,s_016um_00079_00303-1,"POLYGON ((16988.169 19798.090, 16987.914 19799...",ID_369002
s_016um_00079_00307-1,s_016um_00079_00307-1,0,79,307,19782.526142,17232.106724,s_016um_00079_00307-1,"POLYGON ((17247.307 19774.411, 17247.923 19775...",ID_369003
s_016um_00079_00304-1,s_016um_00079_00304-1,0,79,304,19783.695633,17056.822682,s_016um_00079_00304-1,"POLYGON ((17047.526 19780.869, 17048.075 19781...",ID_369004


In [125]:
pooler.filter()

Detected 0 bins (out of 175561, 0.00%) that pointed to no cell
Detected 79075 bins (out of 175561, 45.04%) that pointed to a cell jointly with another bin/s


AnnData object with n_obs × n_vars = 4973 × 19059
    obs: 'in_tissue', 'array_row', 'array_col', 'pxl_row_in_fullres', 'pxl_col_in_fullres', 'index', 'gdf_index'
    var: 'gene_ids', 'feature_types', 'genome'

In [131]:
pooler.pool()

""
ID_100405
ID_100450
ID_100611
ID_100676
ID_101037
...
ID_99175
ID_99536
ID_99658
ID_99669


In [128]:
pooler.gdf.shape

(369005, 2)